# Lab 3.3. When there is no API: the dams of Spain, from a web page

**Module 2, Session 3. ACS-UPM Diploma in Engineering, Data Science and Artificial Intelligence**

By the end of this notebook you should be able to:

1. Check what a site allows before fetching anything, and fetch politely.
2. Take a table with `read_html` when there is one, and with BeautifulSoup when there is not.
3. Keep the provenance of a scraped page: URL, time, and a hash of what was actually received.

Eighteen minutes. The page is a saved copy of a wiki-style list of Spanish dams; the live URL is
tried first. The dam from session 2, Presa de la Hoz, is in the table: the three sessions meet here.

## 0. Setup

This cell locates the course folder wherever the notebook runs: locally, in Colab from the
course repository, or in Colab from the shared Drive folder. Run it first.

In [ ]:
from pathlib import Path

REPO = "https://github.com/antiafer/acs-upm-mod2-s01.git"   # course repository
DRIVE = "ACS-UPM/Mod2-S01"                                 # folder inside My Drive

def course_folder():
    """Return the folder that contains data/, wherever we are running."""
    here = Path.cwd()
    for base in (here, here.parent):
        if (base / "data").is_dir():
            return base
    try:
        import google.colab                            # noqa: F401
    except ImportError:
        raise FileNotFoundError("No data/ folder next to the notebook or one level up.")
    root = Path("/content/acs-mod2")
    if not (root / "data").is_dir():
        import subprocess
        subprocess.run(["git", "clone", "-q", REPO, str(root)], check=False)
    if (root / "data").is_dir():
        return root
    from google.colab import drive
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive") / DRIVE
    if (root / "data").is_dir():
        return root
    raise FileNotFoundError("Could not find the course folder.")

BASE = course_folder()
DATA = BASE / "data"
WORK = Path("/content") if Path("/content").exists() else Path.cwd()
RAW = WORK / "raw"; RAW.mkdir(exist_ok=True)
print("Course folder:", BASE)
print("Raw copies go to:", RAW)

In [ ]:
import importlib, subprocess, sys

def ensure(pkg, module=None):
    try:
        importlib.import_module(module or pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

for pkg, mod in [("requests", None), ("jsonschema", None), ("beautifulsoup4", "bs4"),
                 ("lxml", None), ("pyarrow", None)]:
    ensure(pkg, mod)

import json, re, hashlib, time, datetime as dt
import pandas as pd, requests
print("pandas", pd.__version__, "| requests", requests.__version__)

### Live or cached

Every call in this notebook first tries the live service. If the service does not answer
(twenty people share one address in the room, and public services rate-limit), the cell loads a
copy of the same response saved in `data/`, and says so. The raw copy is not a fallback only: it
is the first step of the pipeline, and the reason a result can be reproduced next year.

**The cached copies are synthetic.** They reproduce the exact shape of each service's answer
(envelope, field names, numbers as strings, missing keys) but the values are invented for the
exercise. Read any number you compute from them as an exercise result, not as a fact about Madrid.
`SOURCES_session3.md` lists what is faithful and what is not in each file.


In [ ]:
def fetch_or_cache(url, cache, *, headers=None, params=None, timeout=15):
    """GET a URL; on any failure, load the cached response from data/. Returns (json, source)."""
    try:
        r = requests.get(url, headers=headers, params=params, timeout=timeout)
        r.raise_for_status()
        doc = r.json()
        # HTTP 200 does not mean the call worked. AEMET answers 200 and puts the real
        # status inside the body, so an expired download link arrives as a normal
        # response reading {"descripcion": "datos expirados", "estado": 404}. Without
        # this check the error envelope flows on as if it were the data.
        if isinstance(doc, dict) and doc.get("estado") not in (None, 200):
            raise ValueError(f"envelope reports estado {doc['estado']}: {doc.get('descripcion')}")
        stamp = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
        (RAW / f"{cache.stem}_{stamp}.json").write_text(json.dumps(doc, ensure_ascii=False), encoding="utf-8")
        return doc, "live"
    except Exception as e:                            # noqa: BLE001
        print(f"Live call failed ({type(e).__name__}); using the cached copy {cache.name}")
        return json.loads(cache.read_text(encoding="utf-8")), "cache"

print("fetch_or_cache ready")

## 1. Rules before requests

### Exercise 1. What does the site allow?

Every site publishes at its root a `robots.txt` that says which paths automated clients may
visit. Python reads it with `urllib.robotparser`. `robots_example.txt` is the file saved with the
page. Parse it and answer: may our client fetch `/wiki/Anexo:Presas_de_España`? And `/w/index.php`?

In [ ]:
from urllib import robotparser
rp = robotparser.RobotFileParser()
rp.parse((DATA / "robots_example.txt").read_text(encoding="utf-8").splitlines())
UA = "ACS-UPM-course/1.0 (teaching; contact: instructor)"
may_wiki = rp.can_fetch(UA, "/wiki/Anexo:Presas_de_España")
may_w = rp.can_fetch(UA, "/w/index.php?title=Special:Export")
delay = rp.crawl_delay("*")
print("wiki page allowed :", may_wiki)
print("/w/ path allowed  :", may_w)
print("crawl delay       :", delay, "s")

In [ ]:
assert may_wiki is True and may_w is False
print("Checks passed. The rule is read before the first request, not after the complaint.")

### Exercise 2. Fetch politely, keep the evidence

A polite client identifies itself with a `User-Agent`, sets a timeout, and waits between
requests. And it keeps what it received: the raw HTML, the time, and a SHA-256 hash of the bytes,
so that a later analysis can show what the page said when it was read.

Fetch the page (live first, then the saved copy) into `html`, and build the `evidence` dict.

In [ ]:
LIVE_URL = "https://es.wikipedia.org/wiki/Anexo:Presas_de_Espa%C3%B1a"
SAVED = DATA / "presas_espana.html"
try:
    r = requests.get(LIVE_URL, headers={"User-Agent": UA}, timeout=15)
    r.raise_for_status(); html, source = r.text, "live"
    time.sleep(delay or 1)
except Exception as e:                                                        # noqa: BLE001
    print(f"Live fetch failed ({type(e).__name__}); using the saved copy")
    html, source = SAVED.read_text(encoding="utf-8"), "saved copy"
raw_path = RAW / f"presas_{dt.datetime.now(dt.timezone.utc):%Y%m%dT%H%M%SZ}.html"
raw_path.write_text(html, encoding="utf-8")
evidence = {"url": LIVE_URL, "fetched_at": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
            "source": source, "sha256": hashlib.sha256(html.encode("utf-8")).hexdigest(), "raw_copy": raw_path.name}
for k, v in evidence.items():
    print(f"{k:<11} {v}")

In [ ]:
assert set(evidence) >= {"url", "fetched_at", "sha256", "raw_copy"} and len(evidence["sha256"]) == 64
assert "Presa" in html
print("Checks passed: the page is on disk with its time and its fingerprint.")

## 2. The simplest route first

### Exercise 3. A real table: read_html

If the page has a real `table` element, `read_html` does the work, exactly as with the bulletin on
Friday. Extract the dams table into `dams`, remove the footnote markers from the names, and make
sure height and capacity are numbers.

In [ ]:
from io import StringIO
dams = pd.read_html(StringIO(html), match="Capacidad", thousands=".", decimal=",")[0]
dams.columns = ["dam", "river", "province", "height_m", "capacity_hm3", "year"]
dams["dam"] = dams["dam"].str.replace(r"\[\d+\]", "", regex=True).str.strip()
print(dams.shape)
dams.head(4)

In [ ]:
assert len(dams) == 18 and dams["capacity_hm3"].dtype.kind == "f"
assert not dams["dam"].str.contains(r"\[").any(), "Footnote markers remain in the names"
assert (dams["province"] == "Madrid").sum() == 3
print("Checks passed: 18 dams, 3 of them in Madrid.")

## 3. When the data are not in a table

### Exercise 4. Selectors with BeautifulSoup

The summary box at the top of the page is not a table: three `p.dato` paragraphs, each with a
`span.etiqueta` and a `span.valor`. `read_html` cannot see it. Parse the page with BeautifulSoup,
select those paragraphs with a CSS selector, and build the dictionary `summary`
(label to value).

In [ ]:
from bs4 import BeautifulSoup
soup = BeautifulSoup(html, "lxml")
summary = {p.select_one("span.etiqueta").get_text(strip=True): p.select_one("span.valor").get_text(strip=True)
           for p in soup.select("div#resumen p.dato")}
summary

In [ ]:
assert len(summary) == 3 and "Presas inventariadas" in summary
assert summary["Presas inventariadas"] == "1.226"
print("Checks passed. Note that the value is still text with a thousands point: typing is your job, as always.")

### Exercise 5. Lists and links

The section *Presas por comunidad autónoma* is a list: each `li` has a link with the region and a
`span.conteo` with the count in brackets. Build `regions` with columns `region`, `count`, `href`.
Links are data too: they are how a scraper walks from a list page to each detail page.

In [ ]:
rows = []
for li in soup.select("ul.lista-comunidades li"):
    a = li.select_one("a")
    n = int(li.select_one("span.conteo").get_text(strip=True).strip("()"))
    rows.append({"region": a.get_text(strip=True), "count": n, "href": a["href"]})
regions = pd.DataFrame(rows)
regions

In [ ]:
assert len(regions) == 5 and regions["count"].dtype.kind == "i"
assert regions.loc[regions["region"].str.contains("Madrid"), "count"].iloc[0] == 31
print("Checks passed: five regions, with the link that would lead to each detail page.")

## 4. Three sessions, one dam

### Exercise 6. Close the loop with session 2

On Friday the reservoir bulletin gave Presa de la Hoz a capacity of 96.4 hm³. Find the same dam in
the scraped table and compare. Then attach the evidence to the table: every row gets the URL,
the fetch time and the hash, as columns.

In [ ]:
BULLETIN_CAPACITY = 96.4
hoz = dams.loc[dams["dam"] == "Presa de la Hoz"].iloc[0]
scraped_capacity = float(hoz["capacity_hm3"])
dams_prov = dams.assign(source_url=evidence["url"], fetched_at=evidence["fetched_at"], sha256=evidence["sha256"])
print(f"Bulletin: {BULLETIN_CAPACITY} hm3 | scraped: {scraped_capacity} hm3")
dams_prov.head(2)

In [ ]:
assert abs(scraped_capacity - BULLETIN_CAPACITY) < 0.05
assert {"source_url", "fetched_at", "sha256"} <= set(dams_prov.columns)
print("Checks passed: two sources agree, and the table carries its own evidence.")

### Extension A. The page that is not what it looks like

Some pages carry no data in the HTML at all: a script builds the table in the browser after
loading. A plain request sees an empty container and a `script` tag. Two ways out: read the JSON
inside the script if it is there, or drive a real browser (Playwright, Selenium), which is a
different course. Extract the JSON from the snippet below.

In [ ]:
js_page = """<div id="app"></div>
<script>window.__DATA__ = {"dams": [{"name": "Presa de la Hoz", "capacity_hm3": 96.4}]};</script>"""
m = re.search(r"window\.__DATA__\s*=\s*(\{.*?\});", js_page, re.S)
embedded = json.loads(m.group(1))
assert embedded["dams"][0]["capacity_hm3"] == 96.4
print("Checks passed:", embedded)

---

## Take-aways

- Rules first: robots.txt, terms of use, personal data. Then a polite client with a name and a delay.
- Keep the evidence: raw HTML, time, hash. A scraped number without them cannot be checked.
- `read_html` when there is a table; BeautifulSoup selectors when there is not; a browser only when the page is built by scripts.
- Scraping is the last route, after an API and a bulk download. It is also the most fragile: the page changes, the code breaks.

**References.** Mitchell, *Web Scraping with Python*, 3rd ed. (O'Reilly, 2024). IETF RFC 9309
(Robots Exclusion Protocol). Dogucu and Çetinkaya-Rundel (2021), Web scraping in the statistics and
data science curriculum, *Journal of Statistics and Data Science Education*.